# GRPO from Scratch: Group-Relative Advantage Without a Critic

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/reinforcement-learning/grpo_from_scratch.ipynb)

Companion notebook to [the blog post](https://sesen.ai/blog/grpo-from-scratch).

GRPO (Group Relative Policy Optimization) is the algorithm behind DeepSeek-R1's
reinforcement learning stage. It removes PPO's critic and replaces it with one line:
sample a group of answers to the same prompt, and standardise their rewards within
the group.

We build it three times, each finishing in seconds on a CPU:

1. **A bandit**, where the true advantage is known in closed form, so we can check
   the estimate is right
2. **A maze**, racing GRPO against PPO and REINFORCE on equal terms
3. **Two-digit arithmetic** with a right/wrong verifier, which is LLM post-training
   in miniature

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from collections import deque

torch.manual_seed(0)
np.random.seed(0)
print("torch", torch.__version__)

## 1. The whole algorithm

Everything GRPO adds to a policy gradient is this function. Subtracting the group
mean is the baseline that replaces the critic; dividing by the group standard
deviation puts every prompt on the same scale.

The `eps` matters more than it looks. When every sample in a group scores the same,
the numerator and denominator are both zero. Without `eps` you get a NaN; with it you
get an advantage of exactly zero, meaning the group contributes nothing.

In [ ]:
def group_relative_advantage(rewards, eps=1e-4):
    """rewards: (n_groups, group_size) -> standardised advantages, same shape."""
    mean = rewards.mean(axis=-1, keepdims=True)
    std = rewards.std(axis=-1, keepdims=True)
    return (rewards - mean) / (std + eps)


# A group of eight scores for one prompt
demo = np.array([[0.9, 0.2, 0.75, 0.1, 0.85, 0.35, 0.6, 0.15]])
print("rewards   ", demo.ravel())
print("advantages", np.round(group_relative_advantage(demo).ravel(), 2))

# What happens when the group is unanimous
print("\nunanimous  ", np.round(group_relative_advantage(np.ones((1, 8))).ravel(), 6))

## 2. Quick win: GRPO on a bandit

Five arms, unknown rewards, no critic. The policy should end up on arm 2.

In [ ]:
def softmax(z):
    e = np.exp(z - z.max())
    return e / e.sum()

rng = np.random.default_rng(5)
true_reward = np.array([0.31, 1.10, 0.42, 0.18, 0.72])   # unknown to the agent
logits, G, lr = np.zeros(5), 8, 1.6
history = []

for step in range(14):
    probs = softmax(logits)
    actions = rng.choice(5, size=G, p=probs)              # one group of G samples
    rewards = true_reward[actions] + rng.normal(0, 0.3, G)
    advantage = group_relative_advantage(rewards.reshape(1, -1)).ravel()

    grad = np.zeros(5)                                    # score-function gradient
    for a, adv in zip(actions, advantage):
        onehot = np.zeros(5); onehot[a] = 1.0
        grad += adv * (onehot - probs)
    logits += lr * grad / G
    history.append(softmax(logits).copy())

print("final policy:", np.round(softmax(logits), 3))
print("best arm    :", int(np.argmax(true_reward)) + 1)

plt.figure(figsize=(7, 4))
for a in range(5):
    plt.plot([h[a] for h in history], label=f"$a_{a+1}$",
             lw=2.5 if a == 1 else 1.2)
plt.xlabel("GRPO update"); plt.ylabel(r"$\pi(a \mid s)$")
plt.title("The group collapses onto the best arm"); plt.legend(); plt.show()

## 3. Is the group-relative advantage actually the advantage?

On a contextual bandit the true advantage `A(s,a) = q(s,a) - V(s)` is available in
closed form, so we can check GRPO's estimate against it instead of trusting it.

Two wrinkles to expect. The estimate recovers the advantage divided by `sigma`, the
spread of rewards the policy sees, because of the standardisation. And it carries a
factor of `(G-1)/G`, because sample `i` sits inside the mean it is compared against.

In [ ]:
class ContextualBandit:
    def __init__(self, n_ctx=4, n_act=5, noise=0.30, seed=1):
        rng = np.random.default_rng(seed)
        self.mean_reward = rng.uniform(0.0, 1.0, size=(n_ctx, n_act))
        self.noise, self.n_ctx, self.n_act = noise, n_ctx, n_act

    def sample(self, ctx, actions, rng):
        return self.mean_reward[ctx, actions] + rng.normal(0, self.noise, size=actions.shape)

    def true_advantage(self, ctx, probs):
        q = self.mean_reward[ctx]
        return q - float(probs @ q)

    def reward_std(self, ctx, probs):
        q = self.mean_reward[ctx]
        v = float(probs @ q)
        return float(np.sqrt(probs @ (q - v) ** 2 + self.noise ** 2))


bandit = ContextualBandit()
rng = np.random.default_rng(0)
logits_all = rng.normal(0, 0.6, size=(bandit.n_ctx, bandit.n_act))
G, n_groups = 8, 8000            # blog post uses 40,000; 8,000 is plenty to see it

estimated, truth = [], []
for ctx in range(bandit.n_ctx):
    probs = softmax(logits_all[ctx])
    actions = rng.choice(bandit.n_act, size=(n_groups, G), p=probs)
    rewards = bandit.sample(ctx, actions, rng)
    adv = group_relative_advantage(rewards)
    sigma, shrink = bandit.reward_std(ctx, probs), (G - 1) / G
    for a in range(bandit.n_act):
        hits = actions == a
        if hits.sum() < 500:
            continue
        estimated.append(adv[hits].mean())
        truth.append(bandit.true_advantage(ctx, probs)[a] / sigma * shrink)

estimated, truth = np.array(estimated), np.array(truth)
slope = float(np.linalg.lstsq(truth[:, None], estimated, rcond=None)[0][0])
r2 = 1 - (estimated - slope * truth).var() / estimated.var()
print(f"slope {slope:.3f}   R^2 {r2:.4f}")

plt.figure(figsize=(5.5, 5.5))
lim = [min(truth.min(), estimated.min()) - 0.1, max(truth.max(), estimated.max()) + 0.1]
plt.plot(lim, lim, "--", color="#9ca3af", label="perfect recovery")
plt.scatter(truth, estimated, s=60, color="#2563eb", alpha=0.85, edgecolor="white")
plt.xlabel(r"closed-form $A(s,a)/\sigma \times (G-1)/G$")
plt.ylabel(r"measured $\hat{A}$"); plt.legend(); plt.show()

## 4. GRPO vs PPO vs REINFORCE on a maze

A 6x6 maze, +1 for reaching the goal and -0.02 a step. The environment is vectorised
so a whole group of rollouts steps in lockstep, which keeps this fast on a CPU.

The point to watch is not the final score (all three eventually solve it) but the
number of iterations and the parameter count. PPO carries a critic; GRPO does not.

In [ ]:
GRID = ["......", ".####.", ".#...#", ".#.#..", "...#.#", ".#...G"]
MOVES = np.array([[-1, 0], [1, 0], [0, -1], [0, 1]])


class GridWorld:
    n_actions, max_steps, step_cost, goal_reward = 4, 40, -0.02, 1.0

    def __init__(self, grid=GRID):
        self.walls = np.array([[c == "#" for c in row] for row in grid])
        self.rows, self.cols = self.walls.shape
        self.goal = next((r, c) for r, row in enumerate(grid)
                         for c, ch in enumerate(row) if ch == "G")
        self.free = [(r, c) for r in range(self.rows) for c in range(self.cols)
                     if not self.walls[r, c] and (r, c) != self.goal]
        self.n_states = self.rows * self.cols

    def encode(self, pos):
        idx = pos[:, 0] * self.cols + pos[:, 1]
        out = np.zeros((len(pos), self.n_states), dtype=np.float32)
        out[np.arange(len(pos)), idx] = 1.0
        return out

    def step(self, pos, actions):
        nxt = pos + MOVES[actions]
        nxt[:, 0] = np.clip(nxt[:, 0], 0, self.rows - 1)
        nxt[:, 1] = np.clip(nxt[:, 1], 0, self.cols - 1)
        blocked = self.walls[nxt[:, 0], nxt[:, 1]]
        nxt[blocked] = pos[blocked]
        reached = (nxt[:, 0] == self.goal[0]) & (nxt[:, 1] == self.goal[1])
        reward = np.full(len(pos), self.step_cost, dtype=np.float32)
        reward[reached] += self.goal_reward
        return nxt, reward, reached


def reachable_starts(env):
    dist, queue = {env.goal: 0}, deque([env.goal])
    while queue:
        r, c = queue.popleft()
        for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < env.rows and 0 <= nc < env.cols and not env.walls[nr, nc]:
                if (nr, nc) not in dist:
                    dist[(nr, nc)] = dist[(r, c)] + 1
                    queue.append((nr, nc))
    return [c for c in env.free if c in dist], dist


env = GridWorld()
starts, dist = reachable_starts(env)
optimal = np.mean([env.step_cost * dist[s] + env.goal_reward for s in starts])
print(f"{len(starts)} start cells, mean {np.mean([dist[s] for s in starts]):.1f} steps, "
      f"optimal return {optimal:.3f}")

In [ ]:
class PolicyNet(nn.Module):
    def __init__(self, n_in, n_out, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_in, hidden), nn.Tanh(), nn.Linear(hidden, n_out))
    def forward(self, x):
        return self.net(x)


class ValueNet(nn.Module):
    """PPO's critic: the network GRPO does not need."""
    def __init__(self, n_in, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_in, hidden), nn.Tanh(), nn.Linear(hidden, 1))
    def forward(self, x):
        return self.net(x).squeeze(-1)


def count_params(*ms):
    return sum(p.numel() for m in ms for p in m.parameters())


def rollout(env, policy, starts, rng):
    pos, alive = np.array(starts), np.ones(len(starts), dtype=bool)
    obs_b, act_b, rew_b, mask_b = [], [], [], []
    for _ in range(env.max_steps):
        obs = env.encode(pos)
        with torch.no_grad():
            probs = F.softmax(policy(torch.as_tensor(obs)), dim=-1).numpy()
        u = rng.random((len(starts), 1))
        actions = (probs.cumsum(axis=1) < u).sum(axis=1).clip(0, env.n_actions - 1)
        nxt, reward, reached = env.step(pos, actions)
        obs_b.append(obs); act_b.append(actions)
        rew_b.append(np.where(alive, reward, 0.0)); mask_b.append(alive.copy())
        pos = np.where(alive[:, None], nxt, pos)
        alive = alive & ~reached
        if not alive.any():
            break
    return {"obs": np.stack(obs_b, 1), "actions": np.stack(act_b, 1),
            "rewards": np.stack(rew_b, 1).astype(np.float32), "mask": np.stack(mask_b, 1)}


def logprobs_of(policy, obs, actions):
    logits = policy(torch.as_tensor(obs))
    idx = torch.as_tensor(actions).unsqueeze(-1)
    return torch.log_softmax(logits, -1).gather(-1, idx).squeeze(-1)

### GRPO

Each start cell is a "prompt". The group is the `G` rollouts launched from it. Note
that the advantage has no time index: every step of a trajectory receives the same
scalar, because the reward is awarded to the whole episode.

In [ ]:
def train_grpo(env, starts, iters=250, n_prompts=8, group=8, lr=3e-3,
               clip=0.2, beta_kl=0.01, inner_epochs=3, seed=0):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    policy = PolicyNet(env.n_states, env.n_actions)
    reference = PolicyNet(env.n_states, env.n_actions)
    reference.load_state_dict(policy.state_dict())
    opt = torch.optim.Adam(policy.parameters(), lr=lr)

    curve = []
    for _ in range(iters):
        prompts = [starts[i] for i in rng.integers(0, len(starts), size=n_prompts)]
        batch = [p for p in prompts for _ in range(group)]
        traj = rollout(env, policy, batch, rng)
        obs, actions, mask = traj["obs"], traj["actions"], traj["mask"]
        episode_return = (traj["rewards"] * mask).sum(axis=1)
        curve.append(float(episode_return.mean()))

        # ---- the only GRPO-specific line
        adv = group_relative_advantage(episode_return.reshape(n_prompts, group)).ravel()
        adv_t = torch.as_tensor(adv, dtype=torch.float32).unsqueeze(1)
        mask_t = torch.as_tensor(mask, dtype=torch.float32)

        with torch.no_grad():
            old_logp = logprobs_of(policy, obs, actions)
            ref_logp = logprobs_of(reference, obs, actions)

        for _ in range(inner_epochs):
            logp = logprobs_of(policy, obs, actions)
            ratio = torch.exp(logp - old_logp)
            surrogate = torch.min(ratio * adv_t, ratio.clamp(1 - clip, 1 + clip) * adv_t)
            log_diff = ref_logp - logp                     # k3 KL estimator
            kl = torch.exp(log_diff) - log_diff - 1.0
            loss = -((surrogate - beta_kl * kl) * mask_t).sum() / mask_t.sum()
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(policy.parameters(), 1.0); opt.step()
    return policy, curve

### PPO, with the critic, and plain REINFORCE

In [ ]:
def train_ppo(env, starts, iters=250, n_prompts=8, group=8, lr=3e-3, clip=0.2,
              gamma=0.99, lam=0.95, inner_epochs=3, vf_coef=0.5, seed=0):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    policy, critic = PolicyNet(env.n_states, env.n_actions), ValueNet(env.n_states)
    opt = torch.optim.Adam(list(policy.parameters()) + list(critic.parameters()), lr=lr)

    curve = []
    for _ in range(iters):
        prompts = [starts[i] for i in rng.integers(0, len(starts), size=n_prompts)]
        batch = [p for p in prompts for _ in range(group)]
        traj = rollout(env, policy, batch, rng)
        obs, actions = traj["obs"], traj["actions"]
        rewards, mask = traj["rewards"], traj["mask"]
        curve.append(float((rewards * mask).sum(axis=1).mean()))

        obs_t = torch.as_tensor(obs); mask_t = torch.as_tensor(mask, dtype=torch.float32)
        with torch.no_grad():
            values = (critic(obs_t) * mask_t).numpy()
            old_logp = logprobs_of(policy, obs, actions)

        steps = rewards.shape[1]
        adv = np.zeros_like(rewards); last = np.zeros(len(rewards), dtype=np.float32)
        for t in reversed(range(steps)):                   # GAE
            nxt_v = values[:, t + 1] if t + 1 < steps else np.zeros(len(rewards), np.float32)
            nxt_m = mask[:, t + 1] if t + 1 < steps else np.zeros(len(rewards), bool)
            delta = rewards[:, t] + gamma * nxt_v * nxt_m - values[:, t]
            last = delta + gamma * lam * nxt_m * last
            adv[:, t] = last
        returns = adv + values
        flat = adv[mask]
        adv_t = torch.as_tensor((adv - flat.mean()) / (flat.std() + 1e-8), dtype=torch.float32)
        ret_t = torch.as_tensor(returns, dtype=torch.float32)

        for _ in range(inner_epochs):
            logp = logprobs_of(policy, obs, actions)
            ratio = torch.exp(logp - old_logp)
            surrogate = torch.min(ratio * adv_t, ratio.clamp(1 - clip, 1 + clip) * adv_t)
            v_loss = ((critic(obs_t) - ret_t) ** 2 * mask_t).sum() / mask_t.sum()
            loss = -(surrogate * mask_t).sum() / mask_t.sum() + vf_coef * v_loss
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(list(policy.parameters()) + list(critic.parameters()), 1.0)
            opt.step()
    return policy, critic, curve


def train_reinforce(env, starts, iters=250, n_prompts=8, group=8, lr=3e-3,
                    gamma=0.99, seed=0):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    policy = PolicyNet(env.n_states, env.n_actions)
    opt = torch.optim.Adam(policy.parameters(), lr=lr)
    curve = []
    for _ in range(iters):
        prompts = [starts[i] for i in rng.integers(0, len(starts), size=n_prompts)]
        batch = [p for p in prompts for _ in range(group)]
        traj = rollout(env, policy, batch, rng)
        rewards, mask = traj["rewards"], traj["mask"]
        curve.append(float((rewards * mask).sum(axis=1).mean()))

        returns = np.zeros_like(rewards); running = np.zeros(len(rewards), np.float32)
        for t in reversed(range(rewards.shape[1])):
            running = rewards[:, t] + gamma * running
            returns[:, t] = running
        ret_t = torch.as_tensor(returns * mask, dtype=torch.float32)
        mask_t = torch.as_tensor(mask, dtype=torch.float32)
        logp = logprobs_of(policy, traj["obs"], traj["actions"])
        loss = -((logp * ret_t) * mask_t).sum() / mask_t.sum()
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(policy.parameters(), 1.0); opt.step()
    return policy, curve

In [ ]:
ITERS = 250     # the blog post uses 300 and averages 3 seeds
grpo_p, grpo_c = train_grpo(env, starts, iters=ITERS)
ppo_p, ppo_critic, ppo_c = train_ppo(env, starts, iters=ITERS)
rf_p, rf_c = train_reinforce(env, starts, iters=ITERS)

def iters_to_90(curve):
    smooth = np.convolve(curve, np.ones(20) / 20, mode="valid")
    hit = smooth >= 0.9 * optimal
    return int(np.argmax(hit)) if hit.any() else -1

print(f"{'method':<11}{'params':>9}{'iters to 90%':>15}")
for name, params, curve in (("GRPO", count_params(grpo_p), grpo_c),
                            ("PPO", count_params(ppo_p, ppo_critic), ppo_c),
                            ("REINFORCE", count_params(rf_p), rf_c)):
    print(f"{name:<11}{params:>9,}{iters_to_90(curve):>15}")

plt.figure(figsize=(9, 4.5))
for curve, color, name in ((grpo_c, "#2563eb", "GRPO"), (ppo_c, "#f59e0b", "PPO"),
                           (rf_c, "#9ca3af", "REINFORCE")):
    plt.plot(np.convolve(curve, np.ones(15) / 15, mode="valid"), color=color, lw=2, label=name)
plt.axhline(optimal, color="#111827", ls=":", lw=1.3)
plt.xlabel("iteration"); plt.ylabel("mean episode return")
plt.title("GRPO matches PPO without a critic"); plt.legend(); plt.show()

## 5. The group size is the variance knob

The critic's real job was variance reduction. With it gone, `G` takes over. Measure
the variance of the gradient estimate at a fixed policy and sweep `G`.

Take the snapshot **mid-training**, while the policy is still stochastic. A converged
policy has almost no gradient variance at any `G`, and the trend reads backwards.

In [ ]:
mid_policy, _ = train_grpo(env, starts, iters=15)     # deliberately under-trained
rng = np.random.default_rng(7)
prompt, sizes, REPLICATES = starts[0], (2, 4, 8, 16, 32), 60   # post uses up to 64, 200 reps

variances = []
for g in sizes:
    grads = []
    for _ in range(REPLICATES):
        traj = rollout(env, mid_policy, [prompt] * g, rng)
        ret = (traj["rewards"] * traj["mask"]).sum(axis=1)
        adv = group_relative_advantage(ret.reshape(1, g)).ravel()
        adv_t = torch.as_tensor(adv, dtype=torch.float32).unsqueeze(1)
        mask_t = torch.as_tensor(traj["mask"], dtype=torch.float32)
        logp = logprobs_of(mid_policy, traj["obs"], traj["actions"])
        loss = -((logp * adv_t) * mask_t).sum() / mask_t.sum()
        mid_policy.zero_grad(); loss.backward()
        grads.append(torch.cat([p.grad.flatten() for p in mid_policy.parameters()]).clone().numpy())
    variances.append(np.stack(grads).var(axis=0).sum())

slope = np.polyfit(np.log(sizes), np.log(variances), 1)[0]
print(f"log-log slope {slope:.2f}  (-1.0 would be exactly 1/G)")

plt.figure(figsize=(6.5, 4.2))
plt.loglog(sizes, variances, "-o", color="#2563eb", lw=2, label="measured")
plt.loglog(sizes, variances[0] * sizes[0] / np.array(sizes), "--",
           color="#9ca3af", label="exact 1/G")
plt.xticks(sizes, sizes); plt.xlabel("group size G"); plt.ylabel("gradient variance")
plt.legend(); plt.show()

## 6. A verifiable reward: two-digit arithmetic

Now the setting GRPO was built for. The model sees `a` and `b`, emits a two-digit
answer one token at a time, and a checker returns 1 if it is exactly right and 0
otherwise. No labels reach the RL stage, only that 0 or 1.

The prompt is encoded as two scalars rather than one-hots on purpose. One-hots turn
this into a 100-entry lookup table that memorises perfectly and generalises to
nothing, which makes the held-out measurement meaningless.

In [ ]:
class ArithmeticPolicy(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(2, hidden), nn.Tanh(),
                                   nn.Linear(hidden, hidden), nn.Tanh())
        self.head1 = nn.Linear(hidden, 10)              # tens digit
        self.tok_embed = nn.Embedding(10, 16)
        self.head2 = nn.Linear(hidden + 16, 10)         # units digit, conditioned on the tens

    def forward(self, prompt, first_token=None):
        h = self.trunk(prompt)
        logits1 = self.head1(h)
        if first_token is None:
            return h, logits1
        return logits1, self.head2(torch.cat([h, self.tok_embed(first_token)], dim=-1))


def all_sums():
    a, b = np.meshgrid(np.arange(10), np.arange(10))
    return a.ravel(), b.ravel()

def encode_prompt(a, b):
    return np.stack([a / 9.0, b / 9.0], axis=1).astype(np.float32)

def check_answer(a, b, t1, t2):
    """The verifier. Exact match only."""
    return ((t1 * 10 + t2) == (a + b)).astype(np.float32)

def sample_answers(policy, a, b, greedy=False):
    prompt = torch.as_tensor(encode_prompt(a, b))
    with torch.no_grad():
        _, l1 = policy(prompt)
        t1 = l1.argmax(-1) if greedy else torch.multinomial(F.softmax(l1, -1), 1).squeeze(-1)
        _, l2 = policy(prompt, t1)
        t2 = l2.argmax(-1) if greedy else torch.multinomial(F.softmax(l2, -1), 1).squeeze(-1)
    return t1, t2

def accuracy_on(policy, idx):
    a, b = all_sums(); a, b = a[idx], b[idx]
    t1, t2 = sample_answers(policy, a, b, greedy=True)
    return float(check_answer(a, b, t1.numpy(), t2.numpy()).mean())

In [ ]:
def sft_warmstart(policy, idx, steps=500, lr=3e-3):
    """The cold start: plain supervised learning on a labelled subset."""
    opt = torch.optim.Adam(policy.parameters(), lr=lr)
    a, b = all_sums(); a, b = a[idx], b[idx]
    prompt = torch.as_tensor(encode_prompt(a, b))
    y1, y2 = torch.as_tensor((a + b) // 10), torch.as_tensor((a + b) % 10)
    for _ in range(steps):
        l1, l2 = policy(prompt, y1)
        loss = F.cross_entropy(l1, y1) + F.cross_entropy(l2, y2)
        opt.zero_grad(); loss.backward(); opt.step()


def train_arithmetic_grpo(policy, iters=1200, n_prompts=32, group=8, lr=5e-4,
                          clip=0.2, ent_coef=0.005, inner_epochs=2, seed=0):
    rng = np.random.default_rng(seed)
    opt = torch.optim.Adam(policy.parameters(), lr=lr)
    all_a, all_b = all_sums()
    acc_curve, degen_curve = [], []

    for _ in range(iters):
        pick = rng.integers(0, 100, size=n_prompts).repeat(group)
        a, b = all_a[pick], all_b[pick]
        prompt = torch.as_tensor(encode_prompt(a, b))
        t1, t2 = sample_answers(policy, a, b)

        reward = check_answer(a, b, t1.numpy(), t2.numpy())
        grouped = reward.reshape(n_prompts, group)
        acc_curve.append(float(reward.mean()))
        degen_curve.append(float((grouped.std(axis=1) == 0).mean()))

        adv = torch.as_tensor(group_relative_advantage(grouped).ravel(), dtype=torch.float32)

        def token_logps():
            l1, l2 = policy(prompt, t1)
            lp1 = torch.log_softmax(l1, -1).gather(-1, t1.unsqueeze(-1)).squeeze(-1)
            lp2 = torch.log_softmax(l2, -1).gather(-1, t2.unsqueeze(-1)).squeeze(-1)
            return lp1 + lp2, (l1, l2)

        with torch.no_grad():
            old_logp, _ = token_logps()
        for _ in range(inner_epochs):
            logp, (l1, l2) = token_logps()
            ratio = torch.exp(logp - old_logp)
            surrogate = torch.min(ratio * adv, ratio.clamp(1 - clip, 1 + clip) * adv)
            entropy = sum(-(F.softmax(l, -1) * F.log_softmax(l, -1)).sum(-1) for l in (l1, l2))
            loss = -(surrogate + ent_coef * entropy).mean()
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(policy.parameters(), 1.0); opt.step()
    return acc_curve, degen_curve

### Two runs: with and without a cold start

The first run goes straight to GRPO from random initialisation. The second gets a
short supervised warm start on 40 of the 100 sums first, then runs identical GRPO on
all 100 with only the verifier.

In [ ]:
perm = np.random.default_rng(0).permutation(100)
SFT_IDX, HELD_IDX = perm[:40], perm[40:]
ITERS = 1200

torch.manual_seed(0)
cold = ArithmeticPolicy()
cold_acc, cold_degen = train_arithmetic_grpo(cold, iters=ITERS)

torch.manual_seed(0)
warm = ArithmeticPolicy()
sft_warmstart(warm, SFT_IDX)
post_sft = {k: accuracy_on(warm, v) for k, v in
            (("40 SFT sums", SFT_IDX), ("60 held-out", HELD_IDX), ("all 100", np.arange(100)))}
warm_acc, warm_degen = train_arithmetic_grpo(warm, iters=ITERS)
post_grpo = {k: accuracy_on(warm, v) for k, v in
             (("40 SFT sums", SFT_IDX), ("60 held-out", HELD_IDX), ("all 100", np.arange(100)))}

print(f"GRPO from random init  : {accuracy_on(cold, np.arange(100)):.2f}")
print(f"{'':24}{'after SFT':>12}{'after GRPO':>13}")
for k in post_sft:
    print(f"{k:<24}{post_sft[k]:>12.2f}{post_grpo[k]:>13.2f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
win = 40
smooth = lambda v: np.convolve(v, np.ones(win) / win, mode="valid")

for degen, acc, color, label in ((cold_degen, cold_acc, "#dc2626", "from random init"),
                                 (warm_degen, warm_acc, "#2563eb", "from SFT cold start")):
    ax1.plot(smooth(degen) * 100, color=color, lw=2, label=label)
    ax2.plot(smooth(acc) * 100, color=color, lw=2, label=label)

ax1.set_xlabel("GRPO iteration"); ax1.set_ylabel("degenerate groups (%)"); ax1.set_ylim(0, 104)
ax1.set_title("Groups where every sample scored the same"); ax1.legend()
ax2.set_xlabel("GRPO iteration"); ax2.set_ylabel("sampled accuracy (%)"); ax2.set_ylim(0, 100)
ax2.set_title("What that costs you"); ax2.legend()
plt.tight_layout(); plt.show()

Both runs end with most groups degenerate, and that is why the curves flatten. As a
policy sharpens it answers each prompt the same way every time, its groups become
unanimous, and the learning signal dries up. GRPO's gradient comes entirely from
disagreement inside a group, so the algorithm starves itself as it succeeds.

This is why frontier recipes run RL after pretraining, and usually after a supervised
cold start. A policy that never succeeds has nothing to be relatively better than.

## Exercises

1. **Kill the epsilon.** Set `eps=0` in `group_relative_advantage` and rerun the
   arithmetic training. Find the iteration where the first NaN appears, and relate it
   to the degenerate-group curve.
2. **Mean only, no standard deviation.** Drop the `/ std` and keep the centring. Does
   the maze still train? Compare the gradient-variance sweep with and without it, and
   decide what the normaliser is buying.
3. **Sweep the group size on a real task.** Run the maze at `group=2, 4, 16` at a
   fixed total episode budget (so `n_prompts * group` stays constant). Is it better to
   see many prompts shallowly or few prompts deeply?
4. **Make the reward partial.** Give 0.5 for getting only the units digit right. Does
   the degenerate-group fraction fall? Does final accuracy improve, or does the model
   learn to farm half marks? (Compare with the
   [reward hacking](https://sesen.ai/blog/reward-hacking-reinforcement-learning) post.)
5. **Harder sums.** Extend to three-digit answers by adding a third head. How many
   more GRPO iterations does the extra token of credit assignment cost?